# 01 - Exploratory Data Analysis
**Federated Learning for Educational Analytics** - UCI Student Performance (Mathematics)

Goals of this notebook:
1. Verify the raw file (rows, columns, separator, missing values).
2. Understand every attribute and its scale.
3. Build the binary target `pass = (G3 >= 10)` and look at class balance.
4. See which features relate to the final grade, and in particular how strongly G1/G2 predict G3.
5. Record dataset limitations that must be reported honestly.

All numbers below are computed from the file, nothing is typed in by hand.

In [1]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src import config
from src.data import load

FIG = config.FIGURES_DIR; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
PALETTE = {"pass": "#2a9d8f", "fail": "#e76f51", "neutral": "#457b9d"}

df = load.load_raw_dataset()
print(df.shape)
df.head()

(395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


## 1. Basic verification
The UCI documentation says 395 students and 33 attributes, semicolon separated. We check that and look for missing values.

In [2]:
s = load.summarise(df)
print(f"rows={s.rows} cols={s.columns} missing={s.missing_values}")
print("dtypes:"); print(df.dtypes.value_counts())
assert s.rows == 395 and s.columns == 33 and s.missing_values == 0

rows=395 cols=33 missing=0
dtypes:
str      17
int64    16
Name: count, dtype: int64


## 2. Attribute dictionary
17 categorical (text) columns and 13 numeric scale/count columns, plus the three grades G1, G2, G3 (0-20).

In [3]:
desc = pd.DataFrame({
    "type": ["categorical" if c in load.CATEGORICAL_COLUMNS else ("grade" if c in config.GRADE_COLUMNS else "numeric") for c in df.columns],
    "unique": [df[c].nunique() for c in df.columns],
    "example": [df[c].iloc[0] for c in df.columns],
    "description": [load.FEATURE_DESCRIPTIONS[c] for c in df.columns],
}, index=df.columns)
pd.set_option("display.max_colwidth", 80)
desc

,type,unique,example,description
school,categorical,2,GP,"Student's school (GP = Gabriel Pereira, MS = Mousinho da Silveira)"
sex,categorical,2,F,Student's sex (F / M)
age,numeric,8,18,Age in years (15-22)
address,categorical,2,U,"Home address type (U = urban, R = rural)"
famsize,categorical,2,GT3,"Family size (LE3 = 3 or fewer, GT3 = more than 3)"
Pstatus,categorical,2,A,"Parents' cohabitation status (T = together, A = apart)"
Medu,numeric,5,4,Mother's education (0 none ... 4 higher education)
Fedu,numeric,5,4,Father's education (0 none ... 4 higher education)
Mjob,categorical,5,at_home,"Mother's job (teacher, health, services, at_home, other)"
Fjob,categorical,5,teacher,"Father's job (teacher, health, services, at_home, other)"


In [4]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,395.0,16.70,1.28,15.0,16.0,17.0,18.0,22.0
Medu,395.0,2.75,1.09,0.0,2.0,3.0,4.0,4.0
Fedu,395.0,2.52,1.09,0.0,2.0,2.0,3.0,4.0
traveltime,395.0,1.45,0.70,1.0,1.0,1.0,2.0,4.0
studytime,395.0,2.04,0.84,1.0,1.0,2.0,2.0,4.0
failures,395.0,0.33,0.74,0.0,0.0,0.0,0.0,3.0
famrel,395.0,3.94,0.90,1.0,4.0,4.0,5.0,5.0
freetime,395.0,3.24,1.00,1.0,3.0,3.0,4.0,5.0
goout,395.0,3.11,1.11,1.0,2.0,3.0,4.0,5.0
Dalc,395.0,1.48,0.89,1.0,1.0,1.0,2.0,5.0


## 3. Target: Pass / Fail
Portuguese secondary grades are on a 0-20 scale where 10 is the pass mark, so `pass = G3 >= 10`.
**G3 is used only to build the label. It is never an input feature.**

In [5]:
y = load.make_binary_target(df)
counts = y.value_counts().rename({1: "Pass", 0: "Fail"})
print(counts, f"\npass rate = {y.mean():.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(counts.index, counts.values, color=[PALETTE["pass"], PALETTE["fail"]])
for i, v in enumerate(counts.values): ax[0].text(i, v + 3, str(v), ha="center")
ax[0].set_title("Class distribution (all 395 students)"); ax[0].set_ylabel("students")

ax[1].hist(df.G3, bins=range(0, 22), color=PALETTE["neutral"], edgecolor="white")
ax[1].axvline(config.PASS_THRESHOLD - 0.5, color=PALETTE["fail"], ls="--", label="pass threshold (10)")
ax[1].set_title("Final grade G3"); ax[1].set_xlabel("G3 (0-20)"); ax[1].legend()
plt.tight_layout(); plt.savefig(FIG / "eda_target_distribution.png"); plt.show()

pass
Pass    265
Fail    130
Name: count, dtype: int64 
pass rate = 0.671


**Class imbalance.** About two thirds of students pass. A model that always says "Pass" would already be ~67% accurate, so accuracy alone is a weak measure. We will always report precision, recall and F1 as well, and compare against this trivial baseline.

Note the spike at G3 = 0: 38 students have a final grade of exactly zero, most likely students who did not sit the final exam (the UCI documentation does not state the reason). They all fall in the Fail class.

In [6]:
print("students with G3 == 0:", int((df.G3 == 0).sum()))
bands = load.make_grade_band_target(df).value_counts().reindex([b[0] for b in config.GRADE_BANDS])
print("Grade bands (for the optional 5-class experiment):"); print(bands)

students with G3 == 0: 38
Grade bands (for the optional 5-class experiment):
grade_band
A     40
B     60
C     62
D    103
F    130
Name: count, dtype: int64


## 4. Categorical features vs pass rate
For each categorical attribute, the share of students who pass in each category.

In [7]:
cat_cols = load.CATEGORICAL_COLUMNS
fig, axes = plt.subplots(4, 5, figsize=(18, 12)); axes = axes.ravel()
for ax, col in zip(axes, cat_cols):
    rate = df.groupby(col)["G3"].apply(lambda g: (g >= 10).mean()).sort_index()
    n = df[col].value_counts().reindex(rate.index)
    ax.bar(rate.index.astype(str), rate.values, color=PALETTE["neutral"])
    ax.axhline(y.mean(), color="grey", ls=":", lw=1)
    for i, (r, c) in enumerate(zip(rate.values, n.values)): ax.text(i, r + 0.01, f"n={c}", ha="center", fontsize=8)
    ax.set_ylim(0, 1); ax.set_title(col)
for ax in axes[len(cat_cols):]: ax.axis("off")
fig.suptitle("Pass rate by category (dotted line = overall pass rate)", y=1.0)
plt.tight_layout(); plt.savefig(FIG / "eda_categorical_pass_rates.png", bbox_inches="tight"); plt.show()

## 5. Numeric features: correlation with the final grade

In [8]:
num_cols = load.NUMERIC_COLUMNS + ["G1", "G2"]
corr = df[num_cols + ["G3"]].corr()["G3"].drop("G3").sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
colors = [PALETTE["pass"] if v > 0 else PALETTE["fail"] for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.set_title("Pearson correlation with G3"); ax.set_xlabel("correlation")
plt.tight_layout(); plt.savefig(FIG / "eda_numeric_correlations.png"); plt.show()
corr.round(3)

failures     -0.360
age          -0.162
goout        -0.133
traveltime   -0.117
health       -0.061
Dalc         -0.055
Walc         -0.052
freetime      0.011
absences      0.034
famrel        0.051
studytime     0.098
Fedu          0.152
Medu          0.217
G1            0.801
G2            0.905
Name: G3, dtype: float64

## 6. How strongly do G1 and G2 predict G3?
This matters for the project design. G1 and G2 are earlier-period grades in the same subject and are very highly correlated with G3. A model that uses them is predicting *final* performance **after** mid-year grades are known - it is not an early-warning model.

That is why the project runs two separate experiments:
* **full**: inputs include G1 and G2 (58 features after encoding),
* **early**: G1 and G2 removed (56 features), to see what can be predicted at the start of the year.

In [9]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, g in zip(ax, ["G1", "G2"]):
    a.scatter(df[g] + np.random.uniform(-.2, .2, len(df)), df.G3 + np.random.uniform(-.2, .2, len(df)),
              c=[PALETTE["pass"] if p else PALETTE["fail"] for p in y], alpha=.6, s=18)
    a.plot([0, 20], [0, 20], color="grey", lw=1, ls=":")
    a.set_xlabel(g); a.set_ylabel("G3"); a.set_title(f"{g} vs G3  (r = {df[g].corr(df.G3):.2f})")
plt.tight_layout(); plt.savefig(FIG / "eda_g1_g2_vs_g3.png"); plt.show()

# How often would a simple rule "pass if G2 >= 10" be right?
rule = (df.G2 >= 10).astype(int)
print(f"Rule 'pass if G2 >= 10' agrees with the true label for {(rule == y).mean():.1%} of students")

Rule 'pass if G2 >= 10' agrees with the true label for 91.9% of students


## 7. Full correlation heatmap (numeric columns)

In [10]:
cm = df[num_cols + ["G3"]].corr()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cm))); ax.set_xticklabels(cm.columns, rotation=90)
ax.set_yticks(range(len(cm))); ax.set_yticklabels(cm.columns)
plt.colorbar(im, fraction=0.03); ax.set_title("Correlation matrix")
plt.tight_layout(); plt.savefig(FIG / "eda_correlation_heatmap.png"); plt.show()

## 8. Findings and limitations (to be reported honestly)

**Findings**
* 395 rows, 33 columns, no missing values, so no imputation is needed.
* 265 Pass / 130 Fail (67.1% pass). Always-Pass baseline accuracy is 67.1%.
* `failures` (past class failures) is the strongest negative predictor among the non-grade features; `Medu`, `studytime` are mildly positive.
* G1 and G2 dominate: the trivial rule "pass if G2 >= 10" already matches the true label for 91.9% of students.

**Limitations**
* Very small dataset (395 students, 79 in the test set). One misclassified student changes test accuracy by about 1.3 percentage points, so small differences between models are not statistically meaningful. We repeat experiments with several seeds and report mean and standard deviation.
* Two Portuguese schools, 2005-2006. Results do not generalise to other countries or years.
* Self-reported survey attributes (alcohol use, family relations) are noisy.
* The federated "institutions" in this project are **simulated partitions of this one dataset**, not real colleges.